# 12 — Capstone: Production Agent

## Background

Previous notebooks built the components: function calling, ReAct loops, memory systems, multi-agent patterns, long-horizon planning, evaluation, trust/safety, human-in-the-loop approval, and observability. This capstone assembles them into a single production-grade agent and runs it against a benchmark task suite.

## What You'll Learn

- How all components compose: safety gate -> HITL approval -> memory retrieval -> ReAct -> tracing
- Agent evaluation at the task level: completion rate, tool efficiency, safety violations
- Failure mode taxonomy: which component breaks under which condition
- Configuration surface: how to tune each layer for different deployment contexts

## Why This Matters

Seeing components in isolation is necessary but not sufficient. The composition reveals emergent failure modes that no individual component test catches: the safety gate that overblocks because the memory system returns stale context, the approval queue that deadlocks because the ReAct loop generates irreversible actions faster than reviewers can clear them.

In [ ]:
import time, uuid, json, random
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional
from contextlib import contextmanager
from enum import Enum
import numpy as np

# ── Enums & base types ─────────────────────────────────────────────────────
class SpanKind(str, Enum):
    THOUGHT = 'thought'; TOOL_CALL = 'tool_call'
    OBSERVATION = 'observation'; LLM_CALL = 'llm_call'; AGENT_RUN = 'agent_run'

class SpanStatus(str, Enum):
    OK = 'ok'; ERROR = 'error'

class EscalationReason(str, Enum):
    LOW_CONFIDENCE   = 'low_confidence'
    HIGH_COST_ACTION = 'high_cost_action'
    IRREVERSIBLE     = 'irreversible'
    POLICY_REQUIRED  = 'policy_required'

class ApprovalStatus(str, Enum):
    APPROVED = 'approved'; REJECTED = 'rejected'; TIMEOUT = 'timeout'

print('Types defined.')


In [ ]:
# ── Tracer ─────────────────────────────────────────────────────────────────
@dataclass
class Span:
    span_id: str; trace_id: str; parent_id: Optional[str]
    name: str; kind: SpanKind; start_ns: int
    end_ns: Optional[int] = None
    status: SpanStatus = SpanStatus.OK
    attributes: Dict[str, Any] = field(default_factory=dict)

    def end(self, status=SpanStatus.OK, **attrs):
        self.end_ns = time.time_ns(); self.status = status
        self.attributes.update(attrs)

    @property
    def duration_ms(self): return (self.end_ns - self.start_ns) / 1e6 if self.end_ns else 0.0

class AgentTracer:
    def __init__(self):
        self._traces: Dict[str, List[Span]] = {}
        self._active_trace_id: Optional[str] = None
        self._active_span_id:  Optional[str] = None

    @contextmanager
    def start_trace(self, name: str):
        tid = str(uuid.uuid4())
        self._traces[tid] = []
        self._active_trace_id = tid
        root = self._new_span(name, SpanKind.AGENT_RUN, None)
        try:
            yield root
        finally:
            if root.end_ns is None: root.end()
            self._active_trace_id = None; self._active_span_id = None

    @contextmanager
    def start_span(self, name: str, kind: SpanKind, **attrs):
        parent_id = self._active_span_id
        span = self._new_span(name, kind, parent_id)
        span.attributes.update(attrs)
        prev = self._active_span_id; self._active_span_id = span.span_id
        try:
            yield span
        except Exception as e:
            span.end(SpanStatus.ERROR, error=str(e)); raise
        finally:
            if span.end_ns is None: span.end()
            self._active_span_id = prev

    def _new_span(self, name, kind, parent_id):
        s = Span(str(uuid.uuid4()), self._active_trace_id, parent_id, name, kind, time.time_ns())
        if self._active_trace_id: self._traces[self._active_trace_id].append(s)
        return s

    def get_trace(self, tid): return self._traces.get(tid, [])

print('Tracer defined.')


In [ ]:
# ── Tool registry + safety layer ───────────────────────────────────────────
@dataclass
class ToolDef:
    name: str; description: str
    reversible: bool = True; cost_level: int = 1; policy_gate: bool = False

TOOLS: Dict[str, ToolDef] = {
    'search':      ToolDef('search',      'Web search',             reversible=True,  cost_level=1),
    'read_file':   ToolDef('read_file',   'Read file from disk',    reversible=True,  cost_level=1),
    'write_file':  ToolDef('write_file',  'Write file to disk',     reversible=False, cost_level=2),
    'send_email':  ToolDef('send_email',  'Send email',             reversible=False, cost_level=2, policy_gate=True),
    'run_sql':     ToolDef('run_sql',     'Execute SQL',            reversible=False, cost_level=3, policy_gate=True),
    'calculator':  ToolDef('calculator',  'Evaluate math expression', reversible=True, cost_level=1),
}

def safety_check(tool: str, args: Dict, semantic_score: float) -> Optional[EscalationReason]:
    td = TOOLS.get(tool)
    if td is None: return EscalationReason.LOW_CONFIDENCE
    if td.policy_gate: return EscalationReason.POLICY_REQUIRED
    adj = semantic_score - (td.cost_level - 1) * 0.1 + (0.1 if td.reversible else -0.1)
    if not td.reversible and adj < 0.75: return EscalationReason.IRREVERSIBLE
    if td.cost_level >= 3:               return EscalationReason.HIGH_COST_ACTION
    if adj < 0.60:                       return EscalationReason.LOW_CONFIDENCE
    return None

# Simulated human reviewer: approves if confidence > 0.75
def human_review(tool: str, reason: EscalationReason, conf: float) -> bool:
    time.sleep(0.005)
    return conf > 0.75 and reason != EscalationReason.LOW_CONFIDENCE

print('Tools and safety layer defined.')


In [ ]:
# ── Episodic memory ────────────────────────────────────────────────────────
@dataclass
class MemoryEntry:
    content: str; created_at: float = field(default_factory=time.time)
    relevance: float = 1.0

class EpisodicMemory:
    def __init__(self, max_entries: int = 20):
        self._store: List[MemoryEntry] = []
        self.max_entries = max_entries

    def store(self, content: str, relevance: float = 1.0):
        self._store.append(MemoryEntry(content, relevance=relevance))
        if len(self._store) > self.max_entries:
            self._store.sort(key=lambda e: e.relevance)
            self._store.pop(0)

    def retrieve(self, query: str, k: int = 3) -> List[str]:
        # Stub: keyword overlap as relevance proxy
        query_words = set(query.lower().split())
        scored = [(sum(1 for w in e.content.lower().split() if w in query_words), e)
                  for e in self._store]
        scored.sort(key=lambda x: -x[0])
        return [e.content for _, e in scored[:k]]

    def __len__(self): return len(self._store)

print('EpisodicMemory defined.')


In [ ]:
# ── Production agent ───────────────────────────────────────────────────────
@dataclass
class TaskResult:
    task:           str
    answer:         Optional[str]
    n_steps:        int
    n_tool_calls:   int
    n_escalations:  int
    n_rejections:   int
    n_errors:       int
    success:        bool
    trace_id:       str
    total_ms:       float

class ProductionAgent:
    def __init__(self, max_steps: int = 8):
        self.tracer   = AgentTracer()
        self.memory   = EpisodicMemory()
        self.max_steps = max_steps
        self._rng     = np.random.default_rng(42)

    def _mock_llm(self, task: str, step: int, context: str) -> tuple:
        # Cycles through a task-specific plan
        plans = {
            'research': [
                ('Search for information.', 'search', 'AI news 2025', 0.92),
                ('Read top result.',        'read_file', 'result.html', 0.88),
                ('Done.', 'Final Answer', 'Research summary: ...', 1.0),
            ],
            'report': [
                ('Calculate totals.',  'calculator', '1+2+3', 0.95),
                ('Write report.',      'write_file', 'report.txt', 0.82),
                ('Email report.',      'send_email', 'boss@co.com', 0.90),
                ('Done.', 'Final Answer', 'Report sent.', 1.0),
            ],
            'data': [
                ('Query database.',    'run_sql', 'SELECT * FROM sales', 0.80),
                ('Summarize results.', 'calculator', 'sum([1,2,3])', 0.95),
                ('Done.', 'Final Answer', 'Data analysis complete.', 1.0),
            ],
        }
        key = 'research'
        for k in plans:
            if k in task.lower(): key = k; break
        plan = plans[key]
        idx  = min(step, len(plan) - 1)
        return plan[idx]

    def _execute_tool(self, tool: str, args: str) -> str:
        if random.random() < 0.05:  # 5% tool failure rate
            raise RuntimeError(f'Tool {tool} failed')
        return f'[{tool} result: {args[:40]}]'

    def run(self, task: str) -> TaskResult:
        t0 = time.perf_counter()
        n_tool_calls = 0; n_escalations = 0; n_rejections = 0; n_errors = 0
        final_answer = None; trace_id_capture = None

        with self.tracer.start_trace(f'task:{task[:40]}') as root:
            trace_id_capture = root.trace_id
            context = ' '.join(self.memory.retrieve(task))
            root.attributes['task'] = task
            root.attributes['memory_entries'] = len(self.memory)

            for step in range(self.max_steps):
                with self.tracer.start_span(f'step_{step}', SpanKind.LLM_CALL, step=step) as llm_span:
                    thought, action, action_input, confidence = self._mock_llm(task, step, context)
                    llm_span.attributes.update({'thought': thought, 'action': action, 'confidence': confidence})

                if action == 'Final Answer':
                    final_answer = action_input
                    self.memory.store(f'Task: {task} -> {final_answer[:60]}', relevance=confidence)
                    break

                # Safety check
                n_tool_calls += 1
                esc_reason = safety_check(action, {'input': action_input}, confidence)

                if esc_reason is not None:
                    n_escalations += 1
                    approved = human_review(action, esc_reason, confidence)
                    if not approved:
                        n_rejections += 1
                        obs = f'[Action {action} rejected by reviewer]'
                        with self.tracer.start_span(f'rejected_{step}', SpanKind.OBSERVATION,
                                                   rejected=True, reason=esc_reason.value): pass
                        context += f' [rejected: {action}]'
                        continue

                with self.tracer.start_span(f'tool_{action}_{step}', SpanKind.TOOL_CALL,
                                            tool=action, input=action_input) as tool_span:
                    try:
                        obs = self._execute_tool(action, action_input)
                        tool_span.attributes['output_len'] = len(obs)
                    except Exception as e:
                        obs = f'ERROR: {e}'
                        n_errors += 1
                        tool_span.end(SpanStatus.ERROR, error=str(e))

                with self.tracer.start_span(f'obs_{step}', SpanKind.OBSERVATION, content=obs[:80]): pass
                context += f' {obs}'

        return TaskResult(
            task=task, answer=final_answer, n_steps=step+1,
            n_tool_calls=n_tool_calls, n_escalations=n_escalations,
            n_rejections=n_rejections, n_errors=n_errors,
            success=final_answer is not None,
            trace_id=trace_id_capture,
            total_ms=(time.perf_counter() - t0) * 1000,
        )

print('ProductionAgent defined.')


## Benchmark Task Suite

In [ ]:
agent = ProductionAgent(max_steps=8)

TASKS = [
    'Research the latest AI news and summarize findings',
    'Generate a monthly report and send via email',
    'Research competitor pricing and write analysis',
    'Analyze sales data and report quarterly totals',
    'Research documentation and create a summary report',
    'Write a data analysis and email results to team',
]

results: List[TaskResult] = []
for task in TASKS:
    r = agent.run(task)
    results.append(r)
    status = 'OK' if r.success else 'FAIL'
    print(f'[{status}] {task[:50]:<50} '
          f'steps={r.n_steps} tools={r.n_tool_calls} esc={r.n_escalations} '
          f'rej={r.n_rejections} err={r.n_errors} {r.total_ms:.0f}ms')


In [ ]:
# ── Aggregate metrics ──────────────────────────────────────────────────────
n           = len(results)
n_success   = sum(1 for r in results if r.success)
avg_steps   = sum(r.n_steps for r in results) / n
avg_tools   = sum(r.n_tool_calls for r in results) / n
avg_esc     = sum(r.n_escalations for r in results) / n
avg_rej     = sum(r.n_rejections for r in results) / n
total_err   = sum(r.n_errors for r in results)
avg_ms      = sum(r.total_ms for r in results) / n

print('=== Benchmark Summary ===')
print(f'  Tasks            : {n}')
print(f'  Success rate     : {n_success}/{n} ({n_success/n*100:.0f}%)')
print(f'  Avg steps/task   : {avg_steps:.1f}')
print(f'  Avg tool calls   : {avg_tools:.1f}')
print(f'  Avg escalations  : {avg_esc:.1f}')
print(f'  Avg rejections   : {avg_rej:.1f}')
print(f'  Total tool errors: {total_err}')
print(f'  Avg latency      : {avg_ms:.0f}ms')
print(f'  Memory entries   : {len(agent.memory)}')

# SLOs
print('\nSLO Checks:')
slos = [
    ('success_rate >= 80%',  n_success/n >= 0.80),
    ('avg_steps <= 6',       avg_steps <= 6.0),
    ('escalation_rate <= 50%', avg_esc / avg_tools <= 0.50 if avg_tools else True),
    ('tool_error_rate <= 10%', total_err / max(1, sum(r.n_tool_calls for r in results)) <= 0.10),
]
for label, passed in slos:
    print(f'  [{"PASS" if passed else "FAIL"}] {label}')


## Failure Mode Taxonomy

A quick reference of what breaks under which conditions — drawn from running the stack above.

In [ ]:
FAILURE_MODES = [
    {
        'mode':      'Tool loop',
        'trigger':   'LLM repeatedly selects same tool with different args',
        'detection': 'Trace: tool_call count > 3 for same tool name',
        'mitigation':'Step budget per tool; deduplicate observations in context',
    },
    {
        'mode':      'Approval deadlock',
        'trigger':   'Agent generates irreversible actions faster than queue clears',
        'detection': 'Approval queue depth > N; escalation rate > 60%',
        'mitigation':'Rate-limit irreversible tool calls; auto-reject on queue overflow',
    },
    {
        'mode':      'Memory staleness',
        'trigger':   'Old memory entries mislead planning for new task context',
        'detection': 'Retrieve-then-check: verify memory date/relevance before injecting',
        'mitigation':'TTL on memory entries; relevance decay function',
    },
    {
        'mode':      'Overblocking',
        'trigger':   'Safety gate too aggressive; too many legitimate actions rejected',
        'detection': 'Rejection rate > 30%; task completion rate drops',
        'mitigation':'Calibrate confidence threshold against rejection/error cost tradeoff',
    },
    {
        'mode':      'Context overflow',
        'trigger':   'Scratchpad grows past model context window mid-task',
        'detection': 'Token count approaching model max_tokens',
        'mitigation':'Observation compression; sliding window on scratchpad',
    },
]

print('=== Failure Mode Taxonomy ===')
for i, fm in enumerate(FAILURE_MODES, 1):
    print(f'\n{i}. {fm["mode"]}')
    print(f'   Trigger:    {fm["trigger"]}')
    print(f'   Detection:  {fm["detection"]}')
    print(f'   Mitigation: {fm["mitigation"]}')


## Key Takeaways

- The production agent is a composition of: tracer -> memory retrieval -> LLM call -> safety check -> [HITL gate] -> tool execution -> memory update
- Every layer can fail independently; the trace is the only artifact that captures the full causality chain
- Benchmark against a fixed task suite before each deploy — success rate, tool efficiency, and escalation rate are the three metrics that matter most
- Failure modes (loop, deadlock, staleness, overblocking, overflow) are predictable; instrument for them from day one
- Memory gives the agent continuity across tasks; without TTL and relevance decay it becomes a liability rather than an asset